In [ ]:
import time
import threading
import math
import logging
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import LaserScan
from geometry_msgs.msg import Twist
from robot_wrapper import Robot
from flask import Flask, render_template_string, request, jsonify
from pyngrok import ngrok

# ==========================================
# 1. GLOBAL STATE & ROS2 THREAD
# ==========================================
global_robot_command = "NO HAND DETECTED"
thread_active = True

class LidarObserver(Node):
    def __init__(self, robot_instance):
        super().__init__('lidar_observer')
        self.robot = robot_instance
        self.subscription = self.create_subscription(LaserScan, '/scan', self.avoiding_obstacles, 10)
        self.safety_latch_engaged = False

    def avoiding_obstacles(self, msg):
        ranges = msg.ranges
        if not ranges: return
        total_points = len(ranges)
        cone_size = max(1, int(total_points / 12))
        front_cone = ranges[:cone_size] + ranges[-cone_size:]
        valid_distances = [r for r in front_cone if 0.15 < r <= 0.50 and not math.isinf(r) and not math.isnan(r)]
        
        if valid_distances:
            if not self.safety_latch_engaged:
                closest_obstacle = min(valid_distances)
                self.get_logger().warn(f"🚨 TARGET DETECTED AT {closest_obstacle:.2f}m! SAFETY LATCH ENGAGED! 🚨")
                self.safety_latch_engaged = True
        else:
            if self.safety_latch_engaged:
                self.get_logger().info("✅ PATH CLEAR! Safety latch released.")
                self.safety_latch_engaged = False

def background_robot_executor():
    global global_robot_command, thread_active
    if not rclpy.ok(): rclpy.init()
        
    robot = Robot()
    node = LidarObserver(robot)
    executor = rclpy.executors.SingleThreadedExecutor()
    executor.add_node(node)
    executor.add_node(robot)
    
    last_command = None
    last_command_time = time.time()

    try:
        while rclpy.ok() and thread_active:
            executor.spin_once(timeout_sec=0.01)
            current_cmd = global_robot_command
            current_time = time.time()

            # Safety override
            if node.safety_latch_engaged:
                current_cmd = "NO HAND DETECTED"

            if current_cmd != last_command or (current_time - last_command_time) > 0.4:
                if current_cmd != last_command:
                    print(f"Executing -> Command: {current_cmd}")
                
                if current_cmd in ["NO HAND DETECTED", "STOP"]:
                    robot.set_cmd_vel(0.0, 0.0, 0.0)
                elif current_cmd == "EMERGENCY STOP":
                    robot.set_cmd_vel(0.0, -0.4, 0.5)  # Spin right to clear obstacle
                elif current_cmd == "GO":
                    robot.set_cmd_vel(0.2, 0.0, 0.5)
                elif current_cmd == "BACKWARD":
                    robot.set_cmd_vel(-0.2, 0.0, 0.5)
                elif current_cmd == "LEFT":
                    robot.set_cmd_vel(0.0, 0.2, 0.5)
                elif current_cmd == "RIGHT":
                    robot.set_cmd_vel(0.0, -0.2, 0.5)
                
                last_command = current_cmd
                last_command_time = current_time
            time.sleep(0.02)
    except Exception as e:
        print(f"Thread encountered error: {e}")
    finally:
        executor.remove_node(node)
        executor.remove_node(robot)
        node.destroy_node()

# ==========================================
# 2. FLASK SERVER & HTML FRONTEND
# ==========================================
# Silence Flask's default terminal logging so it doesn't clutter your notebook
log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)

app = Flask(__name__)

html_page = """
<!DOCTYPE html>
<html>
<head>
    <title>Turtlebot Hand Controller</title>
    <style>
        body { 
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; 
            background-color: #121212; 
            color: #e0e0e0; 
            text-align: center; 
            margin-top: 40px; 
        }
        .container { 
            border: 1px solid #333; 
            padding: 30px; 
            border-radius: 16px; 
            background-color: #8100b4; 
            display: inline-block;
            box-shadow: 0 8px 32px rgba(0,0,0,0.5);
        }
        h2 { margin-top: 0; color: #ffffff; font-weight: 500; letter-spacing: 1px; }
        .video-wrapper { 
            position: relative; 
            width: 640px; 
            height: 480px; 
            margin: 0 auto; 
            background: #000; 
            border-radius: 12px;
            overflow: hidden;
            border: 2px solid #f3dcfc;
        }
        video, canvas { position: absolute; left: 0; top: 0; transform: scaleX(-1); }
        #status-container {
            margin-top: 25px;
            padding: 15px;
            background-color: #2a2a2a;
            border-radius: 8px;
            border-left: 5px solid #00ff00;
        }
        #status { font-size: 28px; font-weight: bold; color: #00ff00; margin: 0; }
        .guide { display: flex; justify-content: center; gap: 20px; margin-top: 20px; font-size: 14px; color: #aaa; }
    </style>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/camera_utils/camera_utils.js" crossorigin="anonymous"></script>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/drawing_utils/drawing_utils.js" crossorigin="anonymous"></script>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/hands/hands.js" crossorigin="anonymous"></script>
</head>
<body>
    <div class="container">
        <h2>Turtlebot Hand Tracking Feed</h2>
        <div class="video-wrapper">
            <video id="webcam" autoplay playsinline width="640" height="480"></video>
            <canvas id="output_canvas" width="640" height="480"></canvas>
        </div>
        <div id="status-container">
            <div id="status">⏳ System Starting...</div>
        </div>
        <div class="guide">
            <span>✊ Stop</span> | 
            <span>✌️ Backward</span> | 
            <span>✋ Forward</span> |
            <span> 🫲 Left </span> |
            <span> 🫱 Right </span>
        </div>
    </div>

    <script>
        const video = document.getElementById('webcam');
        const canvas = document.getElementById('output_canvas');
        const ctx = canvas.getContext('2d');
        const statusText = document.getElementById('status');
        let lastSentGesture = "";

        // UI Emoji Mapping
        const iconMap = {
            "STOP": "✊ STOP",
            "BACKWARD": "✌️ BACKWARD",
            "GO": "✋ GO FORWARD",
            "LEFT": "👈 TURN LEFT",
            "RIGHT": "👉 TURN RIGHT",
            "NO HAND DETECTED": "❌ NO HAND DETECTED",
            "HAND DETECTED": "✊ STOP "
        };

        async function initCamera() {
            try {
                const stream = await navigator.mediaDevices.getUserMedia({ video: true });
                video.srcObject = stream;
            } catch (err) {
                statusText.innerText = "Error: Cannot access webcam.";
                statusText.style.color = "#ff4444";
                document.getElementById('status-container').style.borderLeftColor = "#ff4444";
            }
        }

        const hands = new Hands({
            locateFile: (file) => `https://cdn.jsdelivr.net/npm/@mediapipe/hands/${file}`
        });

        hands.setOptions({
            maxNumHands: 1,
            modelComplexity: 1,
            minDetectionConfidence: 0.5,
            minTrackingConfidence: 0.5
        });

        function evaluateGesture(landmarks) {
            const wrist = landmarks[0];
            function hypot(dx, dy) { return Math.sqrt(dx*dx + dy*dy); }
            function isExtended(tipIdx, knuckleIdx) {
                const distTip = hypot(landmarks[tipIdx].x - wrist.x, landmarks[tipIdx].y - wrist.y);
                const distKnuckle = hypot(landmarks[knuckleIdx].x - wrist.x, landmarks[knuckleIdx].y - wrist.y);
                return distTip > distKnuckle * 1.15;
            }
            
            // Check each finger individually
            const indexExt = isExtended(8, 5);
            const middleExt = isExtended(12, 9);
            const ringExt = isExtended(16, 13);
            const pinkyExt = isExtended(20, 17);

            const openFingers = (indexExt?1:0) + (middleExt?1:0) + (ringExt?1:0) + (pinkyExt?1:0);
                                
            const thumbExt = hypot(landmarks[4].x - wrist.x, landmarks[4].y - wrist.y) > 
                             hypot(landmarks[2].x - wrist.x, landmarks[2].y - wrist.y) * 1.1;

            // 1. CLOSED FIST -> STOP
            if (openFingers === 0 && !thumbExt) {
                return "STOP";
            }

            // 2. PEACE SIGN (Index + Middle extended, Ring + Pinky closed) -> BACKWARD
            if (indexExt && middleExt && !ringExt && !pinkyExt) {
                return "BACKWARD";
            }
            
            // 3. OPEN HAND (3+ fingers) -> Directional Control
            if (openFingers >= 3) {
                const fingertipsX = (landmarks[8].x + landmarks[12].x + landmarks[16].x + landmarks[20].x) / 4.0;
                const diffX = fingertipsX - wrist.x;
                if (diffX < -0.05) return "RIGHT";
                if (diffX > 0.05) return "LEFT";
                return "GO";
            }
            
            return "HAND DETECTED";
        }

        hands.onResults((results) => {
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            ctx.drawImage(results.image, 0, 0, canvas.width, canvas.height);
            let currentGesture = "NO HAND DETECTED";
            
            if (results.multiHandLandmarks && results.multiHandLandmarks.length > 0) {
                const landmarks = results.multiHandLandmarks[0];
                drawConnectors(ctx, landmarks, HAND_CONNECTIONS, {color: '#00FF00', lineWidth: 4});
                drawLandmarks(ctx, landmarks, {color: '#FF0000', lineWidth: 2});
                currentGesture = evaluateGesture(landmarks);
            }
            
            // Update UI with mapped Emojis
            statusText.innerText = iconMap[currentGesture] || currentGesture;
            
            // Push exact command string back to the Flask server quietly
            if (currentGesture !== lastSentGesture) {
                fetch('/set_command', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ command: currentGesture })
                });
                lastSentGesture = currentGesture;
            }
        });

        const camera = new Camera(video, {
            onFrame: async () => { await hands.send({image: video}); },
            width: 640, height: 480
        });

        initCamera().then(() => camera.start());
    </script>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(html_page)

@app.route('/set_command', methods=['POST'])
def set_command():
    global global_robot_command
    data = request.json
    if data and 'command' in data:
        global_robot_command = data['command']
    # Return a quick success response so the JS loop doesn't get blocked
    return jsonify({"status": "success"})

# ==========================================
# 3. MAIN EXECUTION (With Global Link)
# ==========================================
if __name__ == "__main__":
    print("\n" + "="*50)
    print("🤖 STARTING HYBRID SERVER SYSTEM")
    
    # Start the ROS2 backend in a background thread
    worker_thread = threading.Thread(target=background_robot_executor)
    worker_thread.daemon = True
    worker_thread.start()
    
    # --- NGROK SETUP ---
    # PASTE YOUR NEW (SECURE) TOKEN INSIDE THE QUOTES BELOW:
    ngrok.set_auth_token("3H8YwqRhxO33Lc0DA6LSKdBdEDk_5f9s37UfgqoZzGmquegr") 
    
    # Open a global tunnel to the Flask port (5000)
    public_url = ngrok.connect(5000).public_url
    
    print(f"🌍 GLOBAL LINK READY: {public_url}")
    print("👉 Open this link on ANY device to control the robot!")
    print("👉 Stop this cell (Square Button) to shut everything down cleanly.")
    print("="*50 + "\n")
    
    try:
        # Run Flask in the main thread (blocks the notebook cell while running)
        app.run(host='0.0.0.0', port=5000, threaded=True)
    except KeyboardInterrupt:
        pass
    finally:
        print("\n🛑 Shutting down server and closing tunnel...")
        thread_active = False
        
        # Clean up ngrok connections
        try:
            ngrok.disconnect(public_url)
            ngrok.kill()
        except:
            pass
            
        worker_thread.join()
        print("✅ Shutdown complete.")


🤖 STARTING HYBRID SERVER SYSTEM
🌍 GLOBAL LINK READY: https://tribunal-brim-selective.ngrok-free.dev
👉 Open this link on ANY device to control the robot!
👉 Stop this cell (Square Button) to shut everything down cleanly.

 * Serving Flask app '__main__'
 * Debug mode: off
Executing -> Command: NO HAND DETECTED
